In [1]:
import os
import json
import random
import itertools
import math

import numpy as np
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
import torchvision
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt
from matplotlib import colormaps
from IPython.display import display, HTML

from models import get_model
from data import get_dataset

# plt.rcParams["font.size"] = 24
# plt.rcParams["font.family"] = "cmr10"

os.environ["TOKENIZERS_PARALLELISM"] = "true"


[nltk_data] Downloading package punkt_tab to /home/xia/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
device = "cuda:7"
model_dir = "/home/xia/claim-xai/configs/swag/base"
model_config = json.load( open(os.path.join(model_dir, "config.json"), "r") )
model_metadata = json.load( open(os.path.join(model_dir, "masker/_metadata.json"), "r") )

In [3]:
ds_train, ds_val, ds_test, num_classes = get_dataset(
    model_config["dataset_name"], 
    data_dir = "../_datasets", 
    splits = ["train", "validation", "test"],
    **model_config["dataset_args"], 
)


model = get_model(
    model_name = model_config["model_name"],
    checkpoint_path = os.path.join(model_dir, model_metadata["best_checkpoint"]),
    **model_config["model_args"]
).eval().to(device)

Some weights of RobertaModel were not initialized from the model checkpoint at FacebookAI/roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


---

In [4]:
# os.environ["TOKENIZERS_PARALLELISM"] = "true"
# correct = 0

# with torch.no_grad():
#     for images, labels in tqdm(DataLoader(ds_test, batch_size=32, num_workers=2)):
#         # images = images.to(device)
#         labels = labels.to(device)
        
#         inputs = model.preprocess(images)
#         logits, _ = model(inputs)

#         correct += (torch.argmax(logits, dim=1) == labels).sum()

# print(correct / len(ds_test))

In [5]:
# loss = 0

# with torch.no_grad():
#     for texts, labels in tqdm( DataLoader(ds_test, batch_size=32, num_workers=2) ):
#         inputs = model.preprocess(texts)
#         logits, weights = model(inputs)
#         logits = logits.cpu()

#         for b in range(len(logits)):
#             print(logits[b].shape)
#             loss += F.cross_entropy(logits[b], labels[b]).item()

# print(loss / len(ds_test))

In [52]:
context, question, options, label = ds_test[90]

with torch.no_grad():
    inputs = model.preprocess([context], [question], [[o] for o in options])
    logits, weights = model(inputs)

tokens = inputs["input_ids"].cpu().squeeze(0)
weights = weights[0].cpu().squeeze(1)
# weights = (weights - weights.min()) / (weights.max() - weights.min())


print(weights.max(), weights.min())
print(f"pred={torch.argmax(logits).item()} label={label}")


tokens = tokens.tolist()
weights = weights.tolist()



# plt.hist(weights)
# plt.yscale("log")
# plt.show()



text = []
scores = []
curr_word = []

for t, w in zip(tokens[1:], weights[1:]):
    tok = model.text_encoder._model.tokenizer.decode(t)
    if tok == "<pad>": break

    if (len(curr_word) != 0) and (tok[0] == ' '):
        text.append("".join([t for t, w in curr_word]).strip())
        scores.append(sum([w for t, w in curr_word]) / len(curr_word))
        curr_word = []
    curr_word.append((tok, w))
    
if len(curr_word) != 0:
    text.append("".join([t for t, w in curr_word]).strip())
    scores.append(sum([w for t, w in curr_word]) / len(curr_word))



def highlighter(word, score):
    color = colormaps["OrRd"](round((score) * 255))
    color = [round(c*255) for c in color]
    return f"<span style='color: {'white' if score > 0.75 else 'black'}; background-color: rgb({color[0]} {color[1]} {color[2]})'>{word}</span>"

display(HTML(
    ' '.join([('<br>' if t in ["[CONTEXT]", "[QUESTION]", "[OPTION0]", "[OPTION1]", "[OPTION2]", "[OPTION3]"] else '') + highlighter(t, s)
    for t, s in zip(text, scores)])
))

tensor(1.) tensor(0.)
pred=0 label=3


In [7]:
# for t in text:
#     print(f"{t:<4}", end=" ")

# print()

# for t, w in zip(text, scores):
#     print(f"{w:<{len(t)}.2f}", end=" ")





for t in tokens:
    if model.text_encoder._model.tokenizer.decode(t) == "<pad>": break
    print(f"{model.text_encoder._model.tokenizer.decode(t):<4}", end=" ")

print()

for t, w in zip(tokens, weights):
    dec_tok = model.text_encoder._model.tokenizer.decode(t)
    if dec_tok == "<pad>": break
    print(f"{w:<{len(dec_tok)}.2f}", end=" ")

<s>  [    CON  TEXT ]     Someone  drives  down  a    street ,     beer  can  in   hand .     [   QUEST ION  ]     He   [   OP   TION 0    ]     spots  a    record  view  of   the  le  eds   players  and  heads  towards  the  living  room .     [   OP   TION 1    ]     passes  a    dead  -    end  street ,     and  stops .     [   OP   TION 2    ]     stops  at   a    set  of   keys  on   the  bench .     [   OP   TION 3    ]     goes  to   a    window ,     squ int  ing   around  looking  at   the  damage .    </s> 
0.51 0.02 0.19 0.41 0.08 0.23     0.54    0.55  0.39 0.56    0.31 0.84  0.88 0.73 0.88  0.56 0.00 0.31  0.17 0.05 0.16 0.29 0.17 0.28 0.37 0.30 0.49   0.33 0.48    0.49  0.28 0.28 0.48 0.50 0.42     0.30 0.59   0.50     0.27 0.49    0.46  0.56 0.49 0.46 0.53 0.56 0.57 0.83    0.64 0.89  0.74 0.89 0.84    0.53 0.63 0.75   0.56 0.42 0.36 0.41 0.53 0.52 0.69   0.66 0.63 0.72 0.62 0.71  0.62 0.55 0.77   0.56 0.43 0.38 0.50 0.68 0.45 0.85  0.79 0.78 0.93    0.66 0.94 0.87 0.78 